# 104人力銀行網站爬蟲練習
- 從104人力銀行網站爬取求職公司資訊

In [1]:
!pip install -U selenium
!pip install webdriver_manager
!pip install fake-useragent

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
import time

# 建立 Chrome 瀏覽器物件
driver = webdriver.Chrome()
driver.maximize_window()


In [23]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

driver = webdriver.Chrome()
driver.maximize_window()
driver.get("https://www.104.com.tw/company/main/?jobsource=tab_job_to_cs")
time.sleep(3)

# 點「地區」
area_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.XPATH, "//button[span[text()='地區']]"))
)
area_button.click()
time.sleep(1)

# 展開「台北市」
taipei_expand_btn = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.XPATH, "//span[text()='台北市']/ancestor::li/button"))
)
driver.execute_script("arguments[0].click();", taipei_expand_btn)
time.sleep(1)

# 勾選中正區、大同區、大安區
area_values = ["6001001001", "6001001002", "6001001005"]
for val in area_values:
    checkbox = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, f"//input[@value='{val}']"))
    )
    driver.execute_script("arguments[0].click();", checkbox)
    time.sleep(1)

# 點「確定」
confirm_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.XPATH, "//button[contains(@class, 'category-picker-btn-primary') and text()='確定']"))
)
confirm_button.click()
time.sleep(1)



# 點「搜尋」
search_btn = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.XPATH, "//button[@type='submit' and contains(text(),'搜尋')]"))
)
driver.execute_script("arguments[0].click();", search_btn)
time.sleep(3)

# 自動滑到底（讓所有公司都載入）
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

# 抓每間公司資訊
company_cards = driver.find_elements(By.CSS_SELECTOR, ".company-lists__item")
company_data = []

for card in company_cards:
    try:
        name_elem = card.find_element(By.CSS_SELECTOR, ".company-name-link a")
        name = name_elem.text
        link = name_elem.get_attribute("href")

        tags = card.find_elements(By.CSS_SELECTOR, ".company-list__infoTags span")
        area = tags[0].text if len(tags) > 0 else ""
        industry = tags[1].text if len(tags) > 1 else ""
        capital = tags[2].text if len(tags) > 2 else ""
        employee = tags[3].text if len(tags) > 3 else ""

        desc_elem = card.find_element(By.CSS_SELECTOR, ".company-list__description")
        description = desc_elem.text.strip()

        jobs_elem = card.find_element(By.CSS_SELECTOR, "a[href*='#info06']")
        jobs_count = jobs_elem.text.replace("查看工作機會(", "").replace(")", "")

        company_data.append({
            "公司名稱": name,
            "公司連結": link,
            "地區": area,
            "產業": industry,
            "資本額": capital,
            "員工數": employee,
            "公司簡介": description,
            "工作機會數": jobs_count
        })

    except Exception as e:
        print("略過一間公司，原因：", e)

# 轉為表格
df = pd.DataFrame(company_data)
print(df.head())

# 存成 CSV
df.to_csv("公司詳細列表.csv", index=False, encoding="utf-8-sig")

driver.quit()


  公司名稱                                               公司連結      地區         產業  \
0       https://www.104.com.tw/company/1a2x6bkzpy?jobs...  台北市大安區  其他投資理財相關業   
1       https://www.104.com.tw/company/1x0o368?jobsour...  台北市大安區         醫院   
2       https://www.104.com.tw/company/1a2x6bmx05?jobs...  台北市大安區        餐館業   
3       https://www.104.com.tw/company/1a2x6bklpg?jobs...  台北市大安區    專門設計相關業   
4       https://www.104.com.tw/company/13pm89so?jobsou...  台北市中正區      電力供應業   

         資本額      員工數                                               公司簡介 工作機會數  
0  資本額5000萬元  員工數125人  摩爾證券投顧是業界最大規模的公司之一，也是合法之投顧公司，為金管會核准之(113)金管投顧新...        
1    資本額暫不提供  員工數300人  宏恩綜合醫院創立於民國54年，是國內第一家採用開放型態經營的綜合醫院，當時僅有52床。 民國...        
2    資本額暫不提供  員工數暫不提供  我們是一間對於美食及創意充滿近乎偏執熱情的公司，由集團轉投資，獨家代理來自日本大阪的“鯛担麵...        
3    資本額暫不提供  員工數暫不提供  我們重視每一位員工，除了有良好工作環境、也提供學習及成長的空間，歡迎優秀的朋友一起加入蝶心設...        
4  資本額4000萬元   員工數40人  程利工程有限公司創立於1992年，主要營業項目為各式高壓電纜線材、接頭的進出口貿易及銷售。 ...        
